# Experiment 14 — AutoGluon + target-encoded AdaBoost residual stack

Layer 1 is the usual Exp 9 AutoGluon ensemble on the best feature set. Layer 2 predicts Layer 1 residuals using AdaBoost and normal target encoding for every categorical feature. Crucially, residual-model training targets are built from chronological out-of-fold Layer 1 predictions.

**Executed result:** Layer 1 scored **RMSE 468.272**, **MAE 340.987**, and **R² 0.4920**. Adding the AdaBoost residual prediction worsened performance to **497.420**, **386.768**, and **0.4268**—a **29.148 RMSE** loss. The residual correction was too aggressive and target-encoded AdaBoost was not retained.

In [1]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
configured_root=os.environ.get('KDDM_PROJECT_ROOT')
roots=([Path(configured_root)] if configured_root else [])+[Path.cwd(),*Path.cwd().parents]
project_root=next((root for root in roots if (root/'exploration/metul/preprocessing.py').exists()),None)
if project_root is None: raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
sys.path.insert(0,str(project_root/'exploration'/'metul'))
from preprocessing import RentalPreprocessor, load_datasets
TARGET='MONTHLY_RENT'; N_SPLITS=3; RANDOM_STATE=42
OOF_TIME_LIMIT=int(os.environ.get('RESIDUAL_OOF_TIME_LIMIT','180'))
FINAL_TIME_LIMIT=int(os.environ.get('AUTOGLUON_TIME_LIMIT','600'))
RUN_ID=os.environ.get('AUTOGLUON_RUN_ID',pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
HYPERPARAMETERS={'GBM':{},'CAT':{}}
# Define the exact Exp 9 best-feature builder without reusing fitted state.
source_nb=json.loads((project_root/'exploration'/'metul'/'exp_14_resale_features_lgbm.ipynb').read_text())
source_cells={cell.get('metadata',{}).get('id'):''.join(cell.get('source',[])) for cell in source_nb['cells']}
exec(source_cells['setup'],globals())
exec(source_cells['exp9-best-feature-transformer'],globals())


In [2]:
_,datasets=load_datasets(project_root/'data')
raw=datasets['train'].copy(); raw['period']=pd.to_datetime(raw['RENT_APPROVAL_DATE']+'-01').dt.to_period('M')
periods=np.sort(raw.period.unique()); cutoff=periods[int(len(periods)*.80)]
raw_fit,raw_valid=raw.loc[raw.period<cutoff].copy(),raw.loc[raw.period>=cutoff].copy()
def best_features(history,future):
    processor=RentalPreprocessor(datasets)
    history_base=processor.fit_transform(history.drop(columns=['period',TARGET],errors='ignore'))
    future_base=processor.transform(future.drop(columns=['period',TARGET],errors='ignore'))
    return make_exp9_best_features(history,future,history_base,future_base)
fit_x,valid_x=best_features(raw_fit,raw_valid)
fit_y=raw_fit[TARGET].astype(float).reset_index(drop=True); valid_y=raw_valid[TARGET].astype(float).reset_index(drop=True)
print(f'Cutoff={cutoff}; fit={fit_x.shape}; validation={valid_x.shape}')


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


In [3]:
def fit_ag(train_x,train_y,path,time_limit):
    return TabularPredictor(label=TARGET,problem_type='regression',eval_metric='root_mean_squared_error',path=str(path)).fit(train_data=train_x.assign(**{TARGET:train_y.to_numpy()}),hyperparameters=HYPERPARAMETERS,presets=['medium_quality','ignore_text'],time_limit=time_limit,verbosity=2)
fit_periods=np.sort(raw_fit.period.unique()); oof_parts=[]
for fold,(early_idx,later_idx) in enumerate(TimeSeriesSplit(n_splits=N_SPLITS).split(fit_periods),1):
    early=raw_fit.loc[raw_fit.period.isin(fit_periods[early_idx])].copy(); later=raw_fit.loc[raw_fit.period.isin(fit_periods[later_idx])].copy()
    early_x,later_x=best_features(early,later)
    model=fit_ag(early_x,early[TARGET].astype(float).reset_index(drop=True),project_root/'exploration'/'metul'/'ag_exp_14_residual_models'/RUN_ID/f'oof_{fold}',OOF_TIME_LIMIT)
    part=later_x.copy(); part['residual_target']=later[TARGET].to_numpy()-model.predict(later_x).astype(float).to_numpy(); part['fold']=fold
    oof_parts.append(part); print(f'OOF fold {fold}: fit={len(early):,}; residual rows={len(later):,}')
oof=pd.concat(oof_parts,ignore_index=True)


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)


[categorical_clean] derived HDB feature | REGION: 5 -> 5 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       2.66 GB / 16.00 GB (16.6%)
Disk Space Avail:   167.74 GB / 460.38 GB (36.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 180s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_14_residual_models/20261013_155235/oof_1"
Train Data Rows:    31565
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting Aut

OOF fold 1: fit=31,565; residual rows=28,514
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hy

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       2.71 GB / 16.00 GB (16.9%)
Disk Space Avail:   167.74 GB / 460.38 GB (36.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 180s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_14_residual_models/20261013_155235/oof_2"
Train Data Rows:    60079
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to pre

OOF fold 2: fit=60,079; residual rows=27,858
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hy

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       2.75 GB / 16.00 GB (17.2%)
Disk Space Avail:   167.73 GB / 460.38 GB (36.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 180s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_14_residual_models/20261013_155235/oof_3"
Train Data Rows:    87937
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to pre

OOF fold 3: fit=87,937; residual rows=30,051


In [4]:
class NormalTargetEncoder:
    def fit(self,frame,target):
        self.columns_=frame.select_dtypes(include=['object','category','string']).columns.tolist(); self.global_mean_=float(target.mean())
        self.maps_={column:pd.DataFrame({column:frame[column].astype(str),TARGET:target.to_numpy()}).groupby(column)[TARGET].mean() for column in self.columns_}; return self
    def transform(self,frame):
        result=frame.drop(columns=self.columns_).copy()
        for column in self.columns_: result[f'te__{column}']=frame[column].astype(str).map(self.maps_[column]).fillna(self.global_mean_).to_numpy()
        return result
residual_features=oof.drop(columns=['residual_target','fold'])
residual_encoder=NormalTargetEncoder().fit(residual_features,oof['residual_target'])
oof_encoded=residual_encoder.transform(residual_features)
# Fit Layer 1 on all outer-fit data, then construct matching encoded Layer 2 validation inputs.
layer1=fit_ag(fit_x,fit_y,project_root/'exploration'/'metul'/'ag_exp_14_residual_models'/RUN_ID/'layer1_full',FINAL_TIME_LIMIT)
layer1_valid=layer1.predict(valid_x).astype(float).clip(lower=0)
valid_encoded=residual_encoder.transform(valid_x)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       2.61 GB / 16.00 GB (16.3%)
Disk Space Avail:   166.68 GB / 460.38 GB (36.2%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_14_residual_models/20261013_155235/layer1_full"
Train Data Rows:    117988
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators

In [5]:
residual_model=Pipeline([('impute',SimpleImputer(strategy='median')),('adaboost',AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=4,min_samples_leaf=25,random_state=RANDOM_STATE),n_estimators=300,learning_rate=.05,loss='linear',random_state=RANDOM_STATE))])
residual_model.fit(oof_encoded,oof['residual_target'])
residual_valid_prediction=residual_model.predict(valid_encoded)
final_prediction=np.clip(layer1_valid.to_numpy()+residual_valid_prediction,a_min=0,a_max=None)
def metric(name,prediction): return {'model':name,'RMSE':mean_squared_error(valid_y,prediction)**.5,'MAE':mean_absolute_error(valid_y,prediction),'R2':r2_score(valid_y,prediction)}
comparison=pd.DataFrame([metric('AutoGluon Layer 1',layer1_valid),metric('AutoGluon + AdaBoost residual',final_prediction)])
display(comparison.sort_values('RMSE'))
display(pd.DataFrame({'residual_target':oof.residual_target,'predicted_residual':residual_model.predict(oof_encoded)}).describe())


,model,RMSE,MAE,R2
0,AutoGluon Layer 1,468.271759,340.986810,0.491986
1,AutoGluon + AdaBoost residual,497.420083,386.767565,0.426774


,residual_target,predicted_residual
count,86423.000000,86423.000000
mean,217.050004,126.323050
std,572.729411,210.878044
min,-3873.494629,-1118.603384
25%,-137.619995,-37.059056
50%,201.413574,155.115703
75%,584.700317,275.975367
max,3838.455078,1911.902598
